# Playa Quinta — Nearshore Wave Climate Analysis

Analysis of the reconstructed offshore-to-nearshore wave record (RBF reconstruction from 150 SWAN cases). Covers time series comparison, directional/energy analysis, storm (POT) analysis, and offshore-nearshore transformation statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from bluemath_tk.distributions.pot import OptimalThreshold
from bluemath_tk.distributions.gpd import GPD

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RHO = 1025      # seawater density [kg/m3]
G = 9.81        # gravity [m/s2]

In [ ]:
df = pd.read_csv("nearshore_reconstructed_150cases_bluemath.csv", parse_dates=['time'], index_col='time')
df['Hs_ratio'] = df['Hs_quinta_reconstructed'] / df['Hs_offshore']
df.head()

## 1. Time series — offshore vs. nearshore

One arbitrary month, shown as two stacked panels so both series stay readable at their own scale. Change `month` to inspect a different period.

In [ ]:
month = "2015-07"
window = df.loc[month]

fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axs[0].plot(window.index, window['Hs_offshore'], color='#4C72B0', linewidth=1)
axs[0].set_ylabel('Hs offshore [m]')
axs[0].set_title(f'Offshore vs. nearshore wave height — {month}')

axs[1].plot(window.index, window['Hs_quinta_reconstructed'], color='#DD8452', linewidth=1)
axs[1].set_ylabel('Hs nearshore [m]')
axs[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 2. Directional energy analysis

Offshore wave power flux (deep-water approximation, $P = \frac{\rho g^2}{64\pi} H_s^2 T_p$), binned by 10° direction sectors, to identify which approach direction carries the most energy.

In [ ]:
df['P_offshore'] = (RHO * G**2 / (64 * np.pi)) * df['Hs_offshore']**2 * df['Tp_offshore'] / 1000  # kW/m

dmin = np.floor(df['Dir'].min() / 10) * 10
dmax = np.ceil(df['Dir'].max() / 10) * 10
dir_bins = np.arange(dmin, dmax + 10, 10)
df['Dir_bin'] = pd.cut(df['Dir'], bins=dir_bins)

energy_by_dir = df.groupby('Dir_bin', observed=True)['P_offshore'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
energy_by_dir.plot(kind='bar', ax=ax, color='#4C72B0', edgecolor='black')
ax.set_ylabel('Mean wave power [kW/m]')
ax.set_xlabel('Direction bin [deg]')
ax.set_title('Mean offshore wave energy by direction')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

most_energetic = energy_by_dir.idxmax()
print(f"most energetic direction bin: {most_energetic}  ({energy_by_dir.max():.1f} kW/m)")

## 3. Monthly energy analysis

Same wave power metric, broken down by calendar month, and by month **and** direction together to see seasonal shifts in which direction dominates.

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly_energy = df.groupby(df.index.month)['P_offshore'].mean()

fig, ax = plt.subplots(figsize=(9, 5))
monthly_energy.plot(kind='bar', ax=ax, color='#55A868', edgecolor='black')
ax.set_xlabel('Month')
ax.set_ylabel('Mean wave power [kW/m]')
ax.set_title('Mean offshore wave energy by month')
ax.set_xticklabels(month_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pivot = df.pivot_table(values='P_offshore', index=df.index.month, columns='Dir_bin', aggfunc='mean', observed=True)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='viridis')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([month_labels[m - 1] for m in pivot.index])
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha='right')
ax.set_xlabel('Direction bin [deg]')
ax.set_ylabel('Month')
ax.set_title('Mean offshore wave energy by month and direction')
plt.colorbar(im, ax=ax, label='kW/m')
plt.tight_layout()
plt.show()

## 4. Storm analysis — Peaks Over Threshold

Independent storm peaks extracted from the nearshore Hs record via an automatically-selected threshold, with declustering (`min_peak_distance`) enforcing a minimum separation between storms.

In [ ]:
hs_series = df['Hs_quinta_reconstructed'].dropna()
values = hs_series.values
time_idx = hs_series.index

ot = OptimalThreshold(values, n0=10, min_peak_distance=24, method='studentized')
threshold, pks, pks_idx = ot.fit()

storms = pd.DataFrame({
    'time': time_idx[pks_idx],
    'Hs_peak': pks,
})
storms['month'] = storms['time'].dt.month
storms['direction'] = df.loc[storms['time'], 'Dir'].values

print(f"threshold: {threshold:.2f} m")
print(f"independent storms identified: {len(storms)}")
storms.head()

In [ ]:
fig, ax = ot.potplot(time=time_idx.values)
ax.set_title('Storm peaks over threshold — nearshore Hs at Playa Quinta')
plt.tight_layout()
plt.show()

In [ ]:
fit_result = GPD.fit(data=pks, threshold=threshold)

n_years = (hs_series.index.max() - hs_series.index.min()).days / 365.25
lam = len(pks) / n_years

return_periods = np.array([1, 2, 5, 10, 25, 50, 100])
p = 1 - 1 / (lam * return_periods)
Hs_return = GPD.qf(p, threshold=threshold, scale=fit_result.scale, shape=fit_result.shape)

return_table = pd.DataFrame({'Return period [yr]': return_periods, 'Hs [m]': Hs_return})
return_table

## 5. Storm analysis — monthly

The same set of storms, broken down by which calendar month they peaked in: how often storms happen, and how severe they are, by month.

In [ ]:
storm_counts_by_month = storms['month'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
storm_counts_by_month.plot(kind='bar', ax=ax, color='#C44E52', edgecolor='black')
ax.set_xlabel('Month')
ax.set_ylabel('Number of storms')
ax.set_title('Storm frequency by month')
ax.set_xticklabels([month_labels[m - 1] for m in storm_counts_by_month.index], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
storms.boxplot(column='Hs_peak', by='month', ax=ax)
ax.set_xlabel('Month')
ax.set_ylabel('Storm peak Hs [m]')
ax.set_title('Storm severity by month')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Where storms come from

Each storm's peak Hs plotted at its offshore approach direction — a compass view of storm origin and severity.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': 'polar'})
theta = np.radians(storms['direction'])
sc = ax.scatter(theta, storms['Hs_peak'], c=storms['Hs_peak'], cmap='inferno',
                 s=45, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_title('Storm arrival direction and peak Hs', pad=20)
plt.colorbar(sc, ax=ax, label='Hs peak [m]', shrink=0.75, pad=0.1)
plt.tight_layout()
plt.show()

## 7. Offshore vs. nearshore transformation

How the nearshore/offshore Hs ratio depends on period and direction. Raw hourly points are binned (to keep the plot readable at this record length) into direction/period groups, plotted as unconnected scatter points with a linear trend line per group; the printed correlation is computed on the full, unbinned hourly record.

In [ ]:
tp_bin_width = 1
df['Tp_bin_c'] = (df['Tp_offshore'] // tp_bin_width) * tp_bin_width + tp_bin_width / 2

grp = df.groupby(['Dir_bin', 'Tp_bin_c'], observed=True)['Hs_ratio'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
dir_cats = df['Dir_bin'].cat.categories
colors = plt.cm.viridis(np.linspace(0, 1, len(dir_cats)))

for (dbin, sub), color in zip(grp.groupby('Dir_bin', observed=True), colors):
    if len(sub) < 3:
        continue
    ax.scatter(sub['Tp_bin_c'], sub['Hs_ratio'], s=25, color=color, label=str(dbin), alpha=0.85)
    slope, intercept, r, p_val, se = stats.linregress(sub['Tp_bin_c'], sub['Hs_ratio'])
    x_line = np.linspace(sub['Tp_bin_c'].min(), sub['Tp_bin_c'].max(), 20)
    ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.2, linestyle='--')

ax.set_xlabel('Tp offshore [s]')
ax.set_ylabel('Hs ratio (nearshore / offshore)')
ax.set_title('Tp vs. Hs ratio by direction bin')
ax.legend(title='Dir bin', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

r_overall, p_overall = stats.pearsonr(df['Tp_offshore'], df['Hs_ratio'])
print(f"overall correlation, Tp vs. ratio: r = {r_overall:.3f}  (p = {p_overall:.2e})")

In [ ]:
tp_group_width = 3
tp_edges = np.arange(np.floor(df['Tp_offshore'].min()), np.ceil(df['Tp_offshore'].max()) + tp_group_width, tp_group_width)
df['Tp_group'] = pd.cut(df['Tp_offshore'], bins=tp_edges)

dir_bin_width = 5
df['Dir_bin_c'] = (df['Dir'] // dir_bin_width) * dir_bin_width + dir_bin_width / 2

grp2 = df.groupby(['Tp_group', 'Dir_bin_c'], observed=True)['Hs_ratio'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
tp_cats = df['Tp_group'].cat.categories
colors2 = plt.cm.plasma(np.linspace(0, 1, len(tp_cats)))

for (tbin, sub), color in zip(grp2.groupby('Tp_group', observed=True), colors2):
    if len(sub) < 3:
        continue
    ax.scatter(sub['Dir_bin_c'], sub['Hs_ratio'], s=25, color=color, label=str(tbin), alpha=0.85)
    slope, intercept, r, p_val, se = stats.linregress(sub['Dir_bin_c'], sub['Hs_ratio'])
    x_line = np.linspace(sub['Dir_bin_c'].min(), sub['Dir_bin_c'].max(), 20)
    ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.2, linestyle='--')

ax.set_xlabel('Direction offshore [deg]')
ax.set_ylabel('Hs ratio (nearshore / offshore)')
ax.set_title('Direction vs. Hs ratio by Tp bin')
ax.legend(title='Tp bin', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

r_overall2, p_overall2 = stats.pearsonr(df['Dir'], df['Hs_ratio'])
print(f"overall correlation, Dir vs. ratio: r = {r_overall2:.3f}  (p = {p_overall2:.2e})")